<a href="https://colab.research.google.com/github/IBM/vLLM-Hook/blob/main/notebooks/demo_spotlight_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Spotlight Your Instructions: Instruction-following with Dynamic Attention Steering

vLLM-Hook is an extensible framework that allows selective access to model internals during inference. This notebook demonstrates **Spotlight**, an inference-time attention steering method that nudges attention toward emphasized instruction spans.

**Paper**: [Venakteswaran and Contractor, EACL 2026](https://aclanthology.org/2026.eacl-long.174/)

Spotlight requires eager execution so the hook can access attention tensors during prefill.


### Installation

Run this setup cell in a fresh Colab GPU runtime before continuing. It clones the repo, installs compatible runtime dependencies, installs `vllm_hook_plugins`, and restarts the runtime after the first install pass.


In [ ]:
# ==============================================================================
# VLLM HOOK SETUP AND DEPENDENCY MANAGER
# ==============================================================================
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = os.environ.get("VLLM_HOOK_REPO_URL", "https://github.com/IBM/vLLM-Hook.git")
REPO_BRANCH = os.environ.get("VLLM_HOOK_REPO_BRANCH", "main")
REPO_DIR = Path(os.environ.get("VLLM_HOOK_REPO_DIR", "/content/vLLM-Hook"))
SETUP_SENTINEL = Path("/content/.vllm_hook_spotlight_setup_done")

def run(cmd, cwd=None):
    print("+ " + " ".join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, check=True)

if IN_COLAB:
    if not REPO_DIR.exists():
        run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR])
    else:
        run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH])
        run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH])
        run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", REPO_BRANCH])

    filtered_req = Path("/tmp/vllm_hook_colab_requirements.txt")
    req = REPO_DIR / "requirement.txt"
    if req.exists():
        keep = []
        for line in req.read_text(encoding="utf-8").splitlines():
            package = line.strip().split("==", 1)[0].split(">=", 1)[0].split("<", 1)[0].strip().lower()
            if package in {"vllm", "torch", "torchvision", "torchaudio"}:
                continue
            keep.append(line)
        filtered_req.write_text("\n".join(keep) + "\n", encoding="utf-8")
        run([sys.executable, "-m", "pip", "install", "-r", filtered_req])

    run([sys.executable, "-m", "pip", "install", "--force-reinstall", "protobuf>=5.29.6,<6.30"])
    run([sys.executable, "-m", "pip", "install", "-U", "uv"])
    run(["uv", "pip", "install", "--system", "--reinstall", "--no-cache", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu121"])
    run(["uv", "pip", "install", "--system", "--reinstall", "--no-cache", "vllm==0.19.0", "--extra-index-url", "https://wheels.vllm.ai/0.19.0/cu121", "--extra-index-url", "https://download.pytorch.org/whl/cu121", "--index-strategy", "unsafe-best-match"])
    run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR / "vllm_hook_plugins"])

    os.chdir(REPO_DIR / "notebooks")
    if not SETUP_SENTINEL.exists():
        SETUP_SENTINEL.write_text("done", encoding="utf-8")
        print("Restarting Colab runtime so the freshly installed libraries are loaded.")
        import IPython
        IPython.Application.instance().kernel.do_shutdown(True)
else:
    print("Not running in Colab; install dependencies from the repository README if needed.")


### Imports & Environment


In [ ]:
import io
import os
import multiprocessing as mp
import sys
from pathlib import Path

import torch
from vllm import SamplingParams
from vllm_hook_plugins import HookLLM, generate_with_spotlight, register_plugins

IN_COLAB = "google.colab" in sys.modules
os.environ["VLLM_USE_V1"] = "1"

if IN_COLAB:
    mp.set_start_method("fork", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "fork"
    os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
    os.environ.setdefault("HF_HOME", "/content/.cache/huggingface")
    os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/content/.cache/huggingface/hub")
    os.makedirs(os.environ["HUGGINGFACE_HUB_CACHE"], exist_ok=True)

    def _patch_fileno(stream, fallback_stream, fallback_fd):
        try:
            stream.fileno()
        except io.UnsupportedOperation:
            def _fileno():
                try:
                    return fallback_stream.fileno()
                except Exception:
                    return fallback_fd
            stream.fileno = _fileno

    _patch_fileno(sys.stdout, sys.__stdout__, 1)
    _patch_fileno(sys.stderr, sys.__stderr__, 2)
else:
    mp.set_start_method("spawn", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

register_plugins()
print("Environment configured")


### Initialize `HookLLM`


In [ ]:
cache_dir = "/content/.cache/vllm-hook" if IN_COLAB else os.path.expanduser("~/.cache/vllm-hook")
model = "Qwen/Qwen2-1.5B-Instruct"

llm = HookLLM(
    model=model,
    worker_name="probe_spotlight",
    download_dir=cache_dir,
    trust_remote_code=True,
    dtype=torch.float16,
    enable_hook=True,
    gpu_memory_utilization=0.5,
    max_model_len=2048,
    max_num_seqs=1,
    enforce_eager=True,
    enable_prefix_caching=False,
    enable_chunked_prefill=False,
    tensor_parallel_size=1,
)

print(f"Model loaded: {model}")
print("Spotlight worker enabled")


### Configure Test Parameters


In [ ]:
prompt = (
    "Return the response for the following as a JSON: "
    "Write a 400 word paragraph on France and its food specifically "
    "focussing on dishes. Include the paragraph and the list of dishes "
    "mentioned the paragraph in seperate fields."
)
emph_strings = ["Return the response for the following as a JSON:"]
alpha = 0.1
temperature = 0.0
max_tokens = 500

sampling_params = SamplingParams(temperature=temperature, max_tokens=max_tokens)


### Generate Baseline


In [ ]:
outputs_baseline = llm.generate(
    prompts=[prompt],
    sampling_params=sampling_params,
    use_hook=False,
)
baseline_text = outputs_baseline[0].outputs[0].text
print(baseline_text)


### Generate With Spotlight


In [ ]:
outputs_spotlight = generate_with_spotlight(
    llm,
    prompts=[prompt],
    emph_strings=emph_strings,
    alpha=alpha,
    sampling_params=sampling_params,
)
spotlight_text = outputs_spotlight[0].outputs[0].text
print(spotlight_text)


### Comparison


In [ ]:
print("=" * 70)
print("COMPARISON")
print("=" * 70)
print(f"Prompt: {prompt}")
print(f"Emphasized span(s): {emph_strings}")
print(f"Alpha: {alpha}")
print("\nBASELINE")
print("-" * 70)
print(baseline_text)
print("\nWITH SPOTLIGHT")
print("-" * 70)
print(spotlight_text)
